# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/disha526/ML-WEEK1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### I am framing this as a ranking task. A fixed needs refresh vs. doesnt split would force an arbitrary cutoff, treating nearly identical pages differently depending on which side of the line they land on. A continuous score avoids that. It captures how much attention a page needs, so a manager can sort pages and work down the list, tackling the highest priority ones first.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### A custom refresh priority score combining trend_direction (40%), avg_position (35%), and days_since_last_update (25%). Decline matters most since it's the strongest urgency signal; position and staleness add visibility and freshness. Extreme values in position/staleness are capped so outlier pages don't distort the score.

### A defined rule (proxy)
 I constructed it, nobody measured "priority" directly. Risk: a real manager might weigh factors differently (e.g., valuing search_volume more than staleness), so my ranking could disagree with expert judgment.

In [21]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized .csv')

# Convert trend_direction into a number: down = urgent, flat = neutral, up = not urgent, stable = neutral
trend_score = df['trend_direction'].map({'down': 1, 'flat': 0.5, 'up': 0, 'stable': 0.5})

# Normalize avg_position — cap extreme outliers (e.g. position 245) so they don't dominate
position_score = df['avg_position'].clip(upper=50) / 50

# Normalize staleness — cap extreme outliers the same way
staleness_score = df['days_since_last_update'].clip(upper=180) / 180

# Combine into one weighted priority score
df['priority_score'] = (0.4 * trend_score) + (0.35 * position_score) + (0.25 * staleness_score)

print(df[['content_id', 'trend_direction', 'avg_position', 'days_since_last_update', 'priority_score']].head(10))
print()
print('priority_score summary:')
print(df['priority_score'].describe())

             content_id trend_direction  avg_position  days_since_last_update  \
0  content_304f48230142            down          10.6                      20   
1  content_a1fb4e703a9e            down          20.3                      25   
2  content_9aa793d4d895            down          36.5                      20   
3  content_331d6c4de07b          stable           6.2                      22   
4  content_d99b7a2d90ca            down          44.0                      14   
5  content_d4084a4bc775            down           8.5                      20   
6  content_9a34b442b552            down           7.0                      20   
7  content_a63219c6e95a          stable          21.2                      22   
8  content_5e6c160719bc            down          46.0                      20   
9  content_c27558df2b0c            down           4.9                     104   

   priority_score  
0        0.501978  
1        0.576822  
2        0.683278  
3        0.273956  
4       

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Metric: % of pages trending "down" in my top 100 ranked pages vs. bottom 100 ranked pages, compared to the dataset average.
### Result: Top 100 = 100% declining, Bottom 100 = 0% declining, dataset average = 54.2%  a 100point gap.

In [22]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized .csv')

trend_score = df['trend_direction'].map({'down': 1, 'flat': 0.5, 'up': 0})
position_score = df['avg_position'].clip(upper=50) / 50
staleness_score = df['days_since_last_update'].clip(upper=180) / 180

df['priority_score'] = (0.4 * trend_score) + (0.35 * position_score) + (0.25 * staleness_score)

df_sorted = df.sort_values('priority_score', ascending=False)
top_100 = df_sorted.head(100)
bottom_100 = df_sorted.tail(100)

overall_pct_down = (df['trend_direction'] == 'down').mean() * 100
top_pct_down = (top_100['trend_direction'] == 'down').mean() * 100
bottom_pct_down = (bottom_100['trend_direction'] == 'down').mean() * 100

print('Overall % trending down:', round(overall_pct_down, 1))
print('Top 100 (highest priority) % trending down:', round(top_pct_down, 1))
print('Bottom 100 (lowest priority) % trending down:', round(bottom_pct_down, 1))
print('Gap:', round(top_pct_down - bottom_pct_down, 1), 'percentage points')

Overall % trending down: 54.2
Top 100 (highest priority) % trending down: 100.0
Bottom 100 (lowest priority) % trending down: 0.0
Gap: 100.0 percentage points


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### One row = one piece of content (one page), identified by content_id, belonging to a specific client's website (client_id).

In [23]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized .csv')

print('One row = one content item (page)')
print('Total rows:', len(df))
print('Unique content_ids:', df['content_id'].nunique())
print('Unique client_ids:', df['client_id'].nunique())
print()
df[['content_id', 'client_id', 'content_type', 'avg_position', 'trend_direction', 'days_since_last_update']].head(5)

One row = one content item (page)
Total rows: 30000
Unique content_ids: 30000
Unique client_ids: 32



,content_id,client_id,content_type,avg_position,trend_direction,days_since_last_update
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,down,20
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,down,25
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,down,20
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,stable,22
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,down,14


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### A fixed if-statement can't weigh how much each factor matters, or handle combinations that don't fit clean either/or buckets. For example: a page trending down but ranking well (position 5) shouldn't be urgent — a simple rule checking trend_direction=='down' alone would wrongly flag it. Meanwhile, a page that's flat but buried (position 45) and 300 days stale clearly needs attention — but a rule requiring trend_direction=='down' AND position>20 would completely miss it. Real pages exist on a spectrum across multiple factors, not clean categories, and hand-writing if/elif branches for every combination of 3+ factors quickly becomes unmanageable. A weighted score lets each factor contribute proportionally instead of forcing an all-or-nothing rule.

In [24]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized .csv')

trend_score = df['trend_direction'].map({'down': 1, 'flat': 0.5, 'up': 0})
position_score = df['avg_position'].clip(upper=50) / 50
staleness_score = df['days_since_last_update'].clip(upper=180) / 180
df['priority_score'] = (0.4 * trend_score) + (0.35 * position_score) + (0.25 * staleness_score)

# Example: a rigid if-statement would treat these two pages very differently than the weighted score does
example_pages = df[
    ((df['trend_direction'] == 'down') & (df['avg_position'] < 10)) |
    ((df['trend_direction'] == 'flat') & (df['avg_position'] > 40) & (df['days_since_last_update'] > 250))
][['content_id', 'trend_direction', 'avg_position', 'days_since_last_update', 'priority_score']]

print(example_pages.head(10))


              content_id trend_direction  avg_position  \
5   content_d4084a4bc775            down           8.5   
6   content_9a34b442b552            down           7.0   
9   content_c27558df2b0c            down           4.9   
16  content_78bd1d4a1d4d            down           8.9   
17  content_761a44afda12            down           7.3   
19  content_af865035b328            down           6.9   
20  content_0d748c484ab1            down           5.6   
22  content_3fb46bec4413            down           3.9   
24  content_0e23e310d404            down           4.1   
25  content_033ae3e7aecf            down           7.2   

    days_since_last_update  priority_score  
5                       20        0.487278  
6                       20        0.476778  
9                      104        0.578744  
16                     104        0.606744  
17                      22        0.481656  
19                      20        0.476078  
20                      20        0.466978  
2

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.